In [7]:
### 프로젝트 루트 경로 설정 및 YAML 설정 파일 로드
# 주피터 노트북이 `notebook/` 폴더 안에서 실행될 때 발생하는 경로 깨짐 문제를 방지하고, 프로젝트 전역 설정(`config/default.yaml`)을 불러옵니다.
import os
import sys
import yaml

# 주피터 노트북 실행 위치에 따른 프로젝트 루트 경로 자동 보정
current_dir = os.getcwd()
if current_dir.endswith("notebook"):
    os.chdir("..")

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# 시스템 전역 설정 파일 로드
with open("config/default.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("설정 로드 완료:", config["generation"])

설정 로드 완료: {'provider': 'openai', 'model': 'gpt-5-nano', 'temperature': 0.1, 'top_p': 0.95, 'max_tokens': 3000, 'history': {'max_turns': 3}}


In [8]:
### 연동 테스트용 Mock 청크 데이터 로드
# 사전에 정의한 가상 제안요청서 청크 데이터(`sample_chunks.jsonl`)를 `SearchResult` 객체로 파싱합니다.
import json
from src.retriever import SearchResult

mock_results = []
file_path = "samples/processed/sample_chunks.jsonl"

# 가상 청크 JSONL 파일을 읽어 SearchResult 객체 리스트로 변환
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        result = SearchResult(
            chunk_id=data["chunk_id"],
            doc_id=data["doc_id"],
            text=data["text"],
            score=0.85,  # 가상 테스트용 임시 유사도 점수
            metadata=data["metadata"]
        )
        mock_results.append(result)

print(f"총 {len(mock_results)}개의 청크 데이터를 성공적으로 불러왔습니다!")

총 3개의 청크 데이터를 성공적으로 불러왔습니다!


In [3]:
### Test Case 1: 문서 내 다중 조건 복합 질문 테스트
# 여러 청크에 나뉘어 있는 정보(주민등록번호 제외 조건, 1년 보관 기간)를 하나의 답변으로 정확히 종합하는지 검증합니다.
from src.rag_engine import generate_answer
question_1 = "기존 시스템에서 사용자 정보 이전할 때 제외해야 하는 항목이 뭐야? 그리고 개인정보 다운로드 기록은 얼마나 보관해야 해?"
response_1 = generate_answer(question_1, mock_results, config)

print(f"Q: {question_1}\n")
print(f"A:\n{response_1['answer']}")

Q: 기존 시스템에서 사용자 정보 이전할 때 제외해야 하는 항목이 뭐야? 그리고 개인정보 다운로드 기록은 얼마나 보관해야 해?

A:
- 기존 시스템에서 사용자 정보를 이전할 때 제외해야 하는 항목: 주민등록번호입니다. [1]

- 개인정보 다운로드 기록 보관 기간: 1년간 보관해야 합니다. [2]

참고 문서
- [1] 2026년 공공 AI 학습지원 플랫폼 구축 사업 (가상디지털진흥원) sample_rfp.txt
- [2] 2026년 공공 AI 학습지원 플랫폼 구축 사업 (가상디지털진흥원) sample_rfp.txt


In [4]:
### Test Case 2: 문서 외 질문에 대한 할루시네이션 방어 테스트
# 제공된 문서에 존재하지 않는 기능에 대해 아는 척 지어내지 않고, 단호하게 "확인할 수 없다"고 차단하는지 검증합니다.
question_2 = "이 플랫폼에 블록체인 연계 기능이 포함되어 있어?"
response_2 = generate_answer(question_2, mock_results, config)

print(f"Q: {question_2}\n")
print(f"A:\n{response_2['answer']}")

Q: 이 플랫폼에 블록체인 연계 기능이 포함되어 있어?

A:
- 제공된 문서에 블록체인 연계 기능의 포함 여부에 대해 명시적으로 다루고 있지 않으므로, 현재 맥락으로는 확인할 수 없습니다. [1][2][3]

참고 문서: sample_rfp.txt (2026년 공공 AI 학습지원 플랫폼 구축 사업)


In [5]:
### Test Case 3: 제안 공고 조건 및 제출 방식 확인 테스트
# RFP의 마감일시와 제한 조건(이메일 제출 불가 등)을 정확하게 추출하여 안내하는지 검증합니다.
question_3 = "제안서 제출 마감일시는 언제이며, 이메일 제출이 가능한가요?"
response_3 = generate_answer(question_3, mock_results, config)

print(f"Q: {question_3}\n")
print(f"A:\n{response_3['answer']}")

Q: 제안서 제출 마감일시는 언제이며, 이메일 제출이 가능한가요?

A:
- 제안서 제출 마감일시는 2026년 8월 14일 17시입니다. [2]
- 이메일 제출은 인정되지 않습니다. 또한 제출은 나라장터를 통한 온라인 제출이며, 전자파일은 PDF 형식으로 등록해야 합니다. [2]

참고 문서: sample_rfp.txt (2026년 공공 AI 학습지원 플랫폼 구축 사업)


In [6]:
### Test Case 4: 시스템 요구사항 및 제약 조건 추출 테스트
# 문서 내에 명시된 기능적 필수 조건(출처 표시, 근거 없을 시 응답 제한)을 누락 없이 요약하는지 검증합니다.
question_4 = "AI 학습 도우미 기능에서 답변을 제공할 때 지켜야 할 필수 조건은 뭐야?"
response_4 = generate_answer(question_4, mock_results, config)

print(f"Q: {question_4}\n")
print(f"A:\n{response_4['answer']}")

Q: AI 학습 도우미 기능에서 답변을 제공할 때 지켜야 할 필수 조건은 뭐야?

A:
- AI 학습 도우미는 답변에 사용한 교육자료의 제목과 위치를 함께 표시해야 한다. [1]
- AI 학습 도우미는 검색된 교육자료에 없는 내용을 사실처럼 생성해서는 안 되며 근거를 찾지 못하면 확인할 수 없다고 답변해야 한다. [2]

참고한 문서: sample_rfp.txt


In [7]:
### Test Case 5: 문서에 없는 부가 정책 질문에 대한 방어 테스트
# 예산이나 보안 외의 사내 복리후생 성격의 질문에 대해 추측하지 않고 근거 부재를 올바르게 안내하는지 검증합니다.
question_5 = "이 프로젝트에 투입되는 개발자들의 식대 지원 기준은 어떻게 돼?"
response_5 = generate_answer(question_5, mock_results, config)

print(f"Q: {question_5}\n")
print(f"A:\n{response_5['answer']}")

Q: 이 프로젝트에 투입되는 개발자들의 식대 지원 기준은 어떻게 돼?

A:
- 요청하신 개발자 식대 지원 기준에 관해, 제공된 문서에서 근거를 찾을 수 없어 확인할 수 없습니다 [1][2][3].

참고한 문서
- sample_rfp.txt (2026년 공공 AI 학습지원 플랫폼 구축 사업)


In [8]:
from src.rag_engine import build_context, SYSTEM_RULE
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-5-nano", max_tokens=1000)
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_RULE + "\n\nContext:\n{context}"),
    ("human", "{question}")
])
chain = prompt | llm

msg = chain.invoke({
    "context": build_context(mock_results),
    "question": "이 플랫폼에 블록체인 연계 기능이 포함되어 있어?"
})
print(msg.response_metadata)
print(repr(msg.content))

{'token_usage': {'completion_tokens': 1000, 'prompt_tokens': 1711, 'total_tokens': 2711, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1000, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E3eewYzfDiAGawIjiIdqi6gdpFpOO', 'service_tier': 'default', 'finish_reason': 'length', 'logprobs': None}
''


In [9]:
### End-to-End 연결 검증: profile별 동작 확인
from api_main import run

for profile in ["baseline", "openai", "local"]:
    print(f"=== profile: {profile} ===")
    response = run("사업 예산과 수행 기간은 어떻게 돼?", profile=profile)
    print(response["answer"])
    print(f"(sources: {len(response['sources'])}건)\n")

=== profile: baseline ===
- 제공된 문서에서 사업 예산과 수행 기간에 대한 구체적 수치를 확인할 수 없습니다. [1][2][3]

참고한 문서:
- 한국연구재단_2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구축 사업
- 인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp
- 국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp
(sources: 3건)

=== profile: openai ===
관련 문서 내용을 찾지 못했습니다. 원본 문서나 검색 조건을 다시 확인해 주세요.
(sources: 0건)

=== profile: local ===
'local' 프로필은 아직 준비되지 않았습니다.
사유: Retrieval 2 담당자가 HuggingFace 임베딩 생성, Chroma 저장·조회, SearchResult 변환을 구현해야 합니다.
(sources: 0건)



In [10]:
# --- 셀 1: 정상 검색 → 답변+출처 확인 ---
from api_main import run, print_result
 
question = "한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?"
response = run(question, profile="baseline")
print_result(question, "baseline", response)

[질문] 한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?
[Profile] baseline

[답변]
- 예산: 130,000,000원 범위 내(VAT 포함) [2]
- 수행 기간: 계약일로부터 3개월(안정화기간 1개월 포함) [2]

참고 문서: 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp [2]

[출처 3건]
1. 한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp | 기관: 한국산업단지공단 | chunk_id: doc_095_chunk_0003 | score: 0.1592
2. 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp | 기관: 한영대학 | chunk_id: doc_001_chunk_0001 | score: 0.1585
3. 한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp | 기관: 한국산업단지공단 | chunk_id: doc_095_chunk_0007 | score: 0.1558


In [11]:
# --- 셀 2: 0건 검색 케이스 확인 ---
question_no_match = "존재하지 않는 사업에 대해 알려줘"
response_no_match = run(
    question_no_match,
    profile="baseline",
    filters={"agency": "존재하지-않는-기관"},
)
print_result(question_no_match, "baseline", response_no_match)

[질문] 존재하지 않는 사업에 대해 알려줘
[Profile] baseline

[답변]
관련 문서 내용을 찾지 못했습니다. 원본 문서나 검색 조건을 다시 확인해 주세요.

[출처 0건]


In [14]:
# --- 이슈 #29: 대화 히스토리·후속 질문 처리 데모 (3턴) ---
import yaml
from pathlib import Path
from src.parser_chunker import load_chunks_jsonl, demo_chunks
from src.retriever_factory import create_retriever
from src.rag_engine import generate_answer, condense_question

with open("config/default.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

chunks_path = Path(config["paths"]["chunks"])
chunks = load_chunks_jsonl(chunks_path) if chunks_path.exists() else demo_chunks()
retriever = create_retriever(chunks, config["retrieval"], "baseline")

history = []

def ask(question, history):
    search_question = condense_question(question, history, config)
    results = retriever.search(search_question, top_k=config["retrieval"]["top_k"], filters=None)
    response = generate_answer(question, results, config, history=history)
    print(f"[질문] {question}")
    if search_question != question:
        print(f"[검색용 재구성 질문] {search_question}")
    print(f"[답변]\n{response['answer']}\n")
    print(f"[출처 {len(response['sources'])}건]")
    for idx, src in enumerate(response["sources"], start=1):
        meta = src.get("metadata", {})
        print(f"{idx}. {meta.get('file_name', '출처 없음')} | 기관: {meta.get('agency', '기관 정보 없음')}")
    print("-" * 60)
    history.append({"question": question, "answer": response["answer"]})

ask("한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산은 얼마야?", history)
ask("그럼 수행 기간은 어떻게 돼?", history)
ask("고려대학교 차세대 포털·학사 정보시스템 구축사업의 예산은 얼마야?", history)

[질문] 한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산은 얼마야?
[답변]
- 예산은 130,000,000원(VAT 포함) 범위 내입니다. [2]

참고한 문서: 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp

[출처 3건]
1. 한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp | 기관: 한국산업단지공단
2. 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp | 기관: 한영대학
3. 한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp | 기관: 한국산업단지공단
------------------------------------------------------------
[질문] 그럼 수행 기간은 어떻게 돼?
[검색용 재구성 질문] 한영대학교 특성화 맞춤형 교육환경 구축 사업의 수행 기간은 얼마나 되며, 시작일과 종료일은 언제인가요?
[답변]
- 수행 기간은 계약일로부터 3개월이며, 안정화기간 1개월이 포함됩니다. [2]

참고한 문서: 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp

[출처 3건]
1. 한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp | 기관: 한국산업단지공단
2. 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp | 기관: 한영대학
3. 한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp | 기관: 한국산업단지공단
------------------------------------------------------------
[질문] 고려대학교 차세대 포털·학사 정보시스템 구축사업의 예산은 얼마야?
[답변]
- 제공된 문서에서 고려대학교 차세대 포털·학사 정보시스템 구축사업의 예산에 대한 구체적 금액은 확인할 수 없습니다. [2]
- 예산 항목의 존재 여부 및 세부 금액은 문서의 사업 범위 및 제안서 구성에서 명시되지 않았습니다. [3]

참고한 문서: 고려대학교_차세대 포털·학사 정보시스템 구

In [5]:
import time
from api_main import load_config, load_chunks
from src.retriever_factory import create_retriever
from src.rag_engine import build_context, build_llm, condense_question, SYSTEM_RULE
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

def diagnose(question: str, profile: str = "openai", config_path: str = "config/default.yaml", n_repeat: int = 3):
    config = load_config(config_path)
    chunks = load_chunks(config["paths"]["chunks"])
    retriever = create_retriever(chunks, config["retrieval"], profile)

    search_question = condense_question(question, None, config)
    results = retriever.search(search_question, top_k=config["retrieval"]["top_k"])
    context = build_context(results)

    llm = build_llm(config)  # StrOutputParser 없이 raw ChatOpenAI
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_RULE + "\n\nContext:\n{context}"),
        MessagesPlaceholder("history", optional=True),
        ("human", "{question}")
    ])
    chain = prompt | llm  # StrOutputParser 제거 -> AIMessage 그대로 받음

    for i in range(n_repeat):
        start = time.perf_counter()
        ai_msg = chain.invoke({"context": context, "question": question, "history": []})
        elapsed = time.perf_counter() - start

        meta = ai_msg.response_metadata
        usage = meta.get("token_usage", {})
        reasoning_tokens = usage.get("completion_tokens_details", {}).get("reasoning_tokens")

        print(f"--- 시도 {i+1} ({elapsed:.2f}s) ---")
        print(f"finish_reason: {meta.get('finish_reason')}")
        print(f"prompt_tokens: {usage.get('prompt_tokens')}, "
              f"completion_tokens: {usage.get('completion_tokens')}, "
              f"reasoning_tokens: {reasoning_tokens}")
        print(f"답변 길이: {len(ai_msg.content)}자")
        print(f"답변 미리보기: {ai_msg.content[:200]!r}")
        print(f"[전체 response_metadata] {meta}")
        print()

diagnose(
    "서울시립대학교의 '대입전형 자료', '학적 정보', '학생 활동 정보' 데이터는 각각 어느 부서에서 담당하여 수집 및 관리하고 있나요?",
    profile="openai",
    n_repeat=3,
)

--- 시도 1 (18.21s) ---
finish_reason: length
prompt_tokens: 3356, completion_tokens: 3000, reasoning_tokens: 3000
답변 길이: 0자
답변 미리보기: ''
[전체 response_metadata] {'token_usage': {'completion_tokens': 3000, 'prompt_tokens': 3356, 'total_tokens': 6356, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 3000, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E4d1KDMiLvwk53XVhgKam1RFkuWZZ', 'service_tier': 'default', 'finish_reason': 'length', 'logprobs': None}

--- 시도 2 (13.90s) ---
finish_reason: stop
prompt_tokens: 3356, completion_tokens: 2678, reasoning_tokens: 2560
답변 길이: 175자
답변 미리보기: '- 대입전형 자료 현황은 담당부서가 입학처에서 수집 및 관리합니다. [4]\n- 학적 정보 현황은 담당부서가 교무처에서 수집 및 관리합니다. [4]\n- 학생 활동 정보 현황은 담당부서가 학생처에서 수집 및 관리합니다. [4]\n\n참고 문서: 서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 

In [11]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import time, yaml
from copy import deepcopy
from langchain_community.callbacks import get_openai_callback
from api_main import run, load_config

BASE_CONFIG_PATH = "config/default.yaml"

PRICES = {  # $ per 1M tokens
    "gpt-5-nano": {"input": 0.05, "output": 0.40},
    "gpt-5-mini": {"input": 0.13, "output": 1.00},
}

CONFIGS = {
    "A_nano_3000": {"model": "gpt-5-nano", "max_tokens": 3000},
    "B_nano_1500": {"model": "gpt-5-nano", "max_tokens": 1500},
    "C_mini_3000":  {"model": "gpt-5-mini", "max_tokens": 3000},
    "D_nano_4500":  {"model": "gpt-5-nano", "max_tokens": 4500},  # reasoning 토큰 소진 대응 상향안
}

QUESTIONS = [
    "한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?",
    "차세대 학사 정보시스템 구축에 대해 알려줘",
]

def make_temp_config(overrides: dict) -> str:
    cfg = deepcopy(load_config(BASE_CONFIG_PATH))
    cfg["generation"].update(overrides)
    tmp_path = f"config/_tmp_{overrides['model']}_{overrides['max_tokens']}.yaml"
    with open(tmp_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, allow_unicode=True)
    return tmp_path

results = []
for cfg_name, overrides in CONFIGS.items():
    tmp_path = make_temp_config(overrides)
    for q in QUESTIONS:
        start = time.perf_counter()
        with get_openai_callback() as cb:
            response = run(q, config_path=tmp_path, profile="openai")
        elapsed = time.perf_counter() - start
        prices = PRICES[overrides["model"]]
        cost = (cb.prompt_tokens * prices["input"] + cb.completion_tokens * prices["output"]) / 1_000_000
        results.append({
            "config": cfg_name, "model": overrides["model"], "max_tokens": overrides["max_tokens"],
            "question": q, "latency_sec": round(elapsed, 2),
            "prompt_tokens": cb.prompt_tokens, "completion_tokens": cb.completion_tokens,
            "answer_len": len(response["answer"]),
            "cost_usd": round(cost, 5), "answer": response["answer"],
        })

import pandas as pd
df = pd.DataFrame(results)
df

,config,model,max_tokens,question,latency_sec,prompt_tokens,completion_tokens,answer_len,cost_usd,answer
0,A_nano_3000,gpt-5-nano,3000,한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?,10.79,3459,1562,138,0.00080,"- 예산: 130,000,000원(VAT 포함)입니다 [1].\n- 수행 기간: 계..."
1,A_nano_3000,gpt-5-nano,3000,차세대 학사 정보시스템 구축에 대해 알려줘,18.51,3235,3000,0,0.00136,
2,B_nano_1500,gpt-5-nano,1500,한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?,8.09,3459,1182,141,0.00065,"- 예산: 예산은 130,000,000원(VAT 포함) 범위 내입니다. [1]\n-..."
3,B_nano_1500,gpt-5-nano,1500,차세대 학사 정보시스템 구축에 대해 알려줘,10.14,3235,1500,0,0.00076,
4,C_mini_3000,gpt-5-mini,3000,한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?,9.36,3459,697,176,0.00115,"- 사업예산: 130,000,000원(부가가치세 포함)입니다. [1] \n- 사업..."
5,C_mini_3000,gpt-5-mini,3000,차세대 학사 정보시스템 구축에 대해 알려줘,23.85,3235,1768,1150,0.00219,요청하신 차세대 학사 정보시스템 구축에 대해 제공된 문서 내용을 기반으로 요약 안내...
6,D_nano_4500,gpt-5-nano,4500,한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?,7.28,3459,993,147,0.00057,"- 예산: 130,000,000원(VAT 포함) 범위 내로 제시되어 있습니다. [1..."
7,D_nano_4500,gpt-5-nano,4500,차세대 학사 정보시스템 구축에 대해 알려줘,25.41,3235,4500,0,0.00196,


In [12]:
NEW_CONFIGS = {
    "E_nano_8000": {"model": "gpt-5-nano", "max_tokens": 8000},   # nano, 예산 훨씬 더 키움
    "F_mini_1500": {"model": "gpt-5-mini", "max_tokens": 1500},   # mini, 예산 줄여봄
}

new_results = []
for cfg_name, overrides in NEW_CONFIGS.items():
    tmp_path = make_temp_config(overrides)
    for q in QUESTIONS:
        start = time.perf_counter()
        with get_openai_callback() as cb:
            response = run(q, config_path=tmp_path, profile="openai")
        elapsed = time.perf_counter() - start
        prices = PRICES[overrides["model"]]
        cost = (cb.prompt_tokens * prices["input"] + cb.completion_tokens * prices["output"]) / 1_000_000
        new_results.append({
            "config": cfg_name, "model": overrides["model"], "max_tokens": overrides["max_tokens"],
            "question": q, "latency_sec": round(elapsed, 2),
            "prompt_tokens": cb.prompt_tokens, "completion_tokens": cb.completion_tokens,
            "answer_len": len(response["answer"]),
            "cost_usd": round(cost, 5), "answer": response["answer"],
        })

df2 = pd.DataFrame(new_results)
df_all = pd.concat([df, df2], ignore_index=True)
df_all

,config,model,max_tokens,question,latency_sec,prompt_tokens,completion_tokens,answer_len,cost_usd,answer
0,A_nano_3000,gpt-5-nano,3000,한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?,10.79,3459,1562,138,0.00080,"- 예산: 130,000,000원(VAT 포함)입니다 [1].\n- 수행 기간: 계..."
1,A_nano_3000,gpt-5-nano,3000,차세대 학사 정보시스템 구축에 대해 알려줘,18.51,3235,3000,0,0.00136,
2,B_nano_1500,gpt-5-nano,1500,한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?,8.09,3459,1182,141,0.00065,"- 예산: 예산은 130,000,000원(VAT 포함) 범위 내입니다. [1]\n-..."
3,B_nano_1500,gpt-5-nano,1500,차세대 학사 정보시스템 구축에 대해 알려줘,10.14,3235,1500,0,0.00076,
4,C_mini_3000,gpt-5-mini,3000,한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?,9.36,3459,697,176,0.00115,"- 사업예산: 130,000,000원(부가가치세 포함)입니다. [1] \n- 사업..."
5,C_mini_3000,gpt-5-mini,3000,차세대 학사 정보시스템 구축에 대해 알려줘,23.85,3235,1768,1150,0.00219,요청하신 차세대 학사 정보시스템 구축에 대해 제공된 문서 내용을 기반으로 요약 안내...
6,D_nano_4500,gpt-5-nano,4500,한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?,7.28,3459,993,147,0.00057,"- 예산: 130,000,000원(VAT 포함) 범위 내로 제시되어 있습니다. [1..."
7,D_nano_4500,gpt-5-nano,4500,차세대 학사 정보시스템 구축에 대해 알려줘,25.41,3235,4500,0,0.00196,
8,E_nano_8000,gpt-5-nano,8000,한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?,8.99,3459,1401,183,0.00073,"- 예산: 130,000,000원 범위 내(부가가치세 포함)입니다. [1]\n- 수..."
9,E_nano_8000,gpt-5-nano,8000,차세대 학사 정보시스템 구축에 대해 알려줘,34.32,3235,6372,811,0.00271,다음은 차세대 학사 정보시스템 구축에 대해 제공된 문서에서 확인할 수 있는 핵심 내...


In [13]:
N_REPEAT = 3

FOCUS_CONFIGS = {
    "B_nano_1500": {"model": "gpt-5-nano", "max_tokens": 1500},
    "A_nano_3000": {"model": "gpt-5-nano", "max_tokens": 3000},
    "D_nano_4500": {"model": "gpt-5-nano", "max_tokens": 4500},
    "E_nano_8000": {"model": "gpt-5-nano", "max_tokens": 8000},
    "F_mini_1500": {"model": "gpt-5-mini", "max_tokens": 1500},
    "C_mini_3000": {"model": "gpt-5-mini", "max_tokens": 3000},
}

Q2 = "차세대 학사 정보시스템 구축에 대해 알려줘"

repeat_results = []
for cfg_name, overrides in FOCUS_CONFIGS.items():
    tmp_path = make_temp_config(overrides)
    for trial in range(N_REPEAT):
        start = time.perf_counter()
        with get_openai_callback() as cb:
            response = run(Q2, config_path=tmp_path, profile="openai")
        elapsed = time.perf_counter() - start
        prices = PRICES[overrides["model"]]
        cost = (cb.prompt_tokens * prices["input"] + cb.completion_tokens * prices["output"]) / 1_000_000
        repeat_results.append({
            "config": cfg_name, "trial": trial + 1,
            "latency_sec": round(elapsed, 2),
            "completion_tokens": cb.completion_tokens,
            "answer_len": len(response["answer"]),
            "success": len(response["answer"]) > 0,
            "cost_usd": round(cost, 5),
        })

df_repeat = pd.DataFrame(repeat_results)
summary = df_repeat.groupby("config").agg(
    success_rate=("success", "mean"),
    avg_completion_tokens=("completion_tokens", "mean"),
    avg_latency_sec=("latency_sec", "mean"),
    avg_cost_usd=("cost_usd", "mean"),
).round(3)
summary

,success_rate,avg_completion_tokens,avg_latency_sec,avg_cost_usd
config,,,,
A_nano_3000,0.000,3000.000,17.507,0.001
B_nano_1500,0.000,1500.000,9.770,0.001
C_mini_3000,1.000,1833.667,19.603,0.002
D_nano_4500,0.667,4252.333,23.667,0.002
E_nano_8000,1.000,4150.333,22.893,0.002
F_mini_1500,0.333,1479.000,16.047,0.002


In [14]:
targets = [
    ("A_nano_3000", "한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?"),
    ("C_mini_3000", "한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼?"),
    ("C_mini_3000", "차세대 학사 정보시스템 구축에 대해 알려줘"),
]

for cfg, q in targets:
    row = df[(df["config"] == cfg) & (df["question"] == q)].iloc[0]
    print(f"=== {cfg} | {q} ===")
    print(row["answer"])
    print()

=== A_nano_3000 | 한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼? ===
- 예산: 130,000,000원(VAT 포함)입니다 [1].
- 수행 기간: 계약일로부터 3개월이며, 안정화기간 1개월 포함입니다 [1].

참고한 문서 출처: 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp [1].

=== C_mini_3000 | 한영대학교 특성화 맞춤형 교육환경 구축 사업의 예산과 수행 기간은 어떻게 돼? ===
- 사업예산: 130,000,000원(부가가치세 포함)입니다. [1]  
- 사업기간: 계약일로부터 3개월(안정화기간 1개월 포함)입니다. [1]

출처: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 (한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp) [1]

=== C_mini_3000 | 차세대 학사 정보시스템 구축에 대해 알려줘 ===
요청하신 차세대 학사 정보시스템 구축에 대해 제공된 문서 내용을 기반으로 요약 안내해 드립니다.

- 사업 목적 및 범위  
  - 학사정보시스템의 정보화 수준 향상 및 표준 업무프로세스 정립을 통해 차세대 시스템에 반영하고자 함[1][5].  
  - 서울·세종 캠퍼스를 아우르는 표준화된 학사행정시스템을 구현하되, 조회/입력 등에서 캠퍼스별 구분 처리가 가능하도록 설계할 것[2].

- 프로세스 재설계(BPR) 및 요구사항  
  - 학사행정 전반에 걸친 업무프로세스 재설계 대상 도출 및 정보화 개선 방안 제시(졸업기준 분석, 대학원 입시·학적 생성 프로세스 점검·재설계 등)[1].  
  - 장학금 관리체계 및 장학 신청→선발→지급·통계 전반의 정보화 향상, 학사원장/수납원장 데이터 정합성 확보를 위한 성적인정 등 학사 프로세스 체계화 포함[1].  
  - 학적·성적·장학 등 신청 프로세스의 진행상태 확인 및 알람 제공 방안 도출, 비학위과정 학생관리 프로세스 재설계 등 요구[1].
